# Read and write Parquet with rugo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mabel-dev/rugo.dev/blob/main/docs/notebooks/parquet.ipynb)

Rugo reads and writes Parquet with no PyArrow, pandas or NumPy. This notebook reads a file's schema without decoding it, reads only the columns and rows you ask for, and writes Parquet back out.

Every example uses the same dataset: 4,630 orbital launch attempts, from Sputnik onwards. More at [rugo.dev](https://rugo.dev).

In [ ]:
%pip install -q rugo

In [ ]:
import os
import urllib.request

if not os.path.exists("space_missions.parquet"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/mabel-dev/rugo.dev/main/docs/data/space_missions.parquet", "space_missions.parquet")
print(f"space_missions.parquet: {os.path.getsize('space_missions.parquet'):,} bytes")

## Read the schema, not the data

`read_metadata` reads the file footer only - the row count and column types, with no column decoded.

In [ ]:
from rugo import parquet

meta = parquet.read_metadata("space_missions.parquet")
print(meta.num_rows, "rows in", meta.num_row_groups, "row group(s)")
for column in meta.schema_columns:
    print(f"{column.name:16} {column.logical_type:14} nullable={column.nullable}")

## Read the whole file

`read_parquet` is a context manager that yields one **morsel** - a batch of typed columns - per row group.

In [ ]:
with parquet.read_parquet("space_missions.parquet") as reader:
    for morsel in reader:
        print(morsel)

## Read only what you need

`columns` picks the columns; `predicates` filters rows. Row groups whose statistics show they can't match are skipped before they're decoded, then the surviving rows are filtered exactly.

In [ ]:
with parquet.read_parquet(
    "space_missions.parquet",
    columns=["Company", "Mission", "Price"],
    predicates=[("Company", "==", "SpaceX"), ("Price", "<", 60.0)],
) as reader:
    spacex = next(iter(reader))

print(spacex.num_rows, "cheap SpaceX launches")
print(spacex)

## Get Python values

A morsel holds typed columns. `column(name).to_pylist()` turns one into a Python list - only do this for the columns you need, since it's the step that creates a Python object per value.

In [ ]:
missions = spacex.column("Mission").to_pylist()
prices = spacex.column("Price").to_pylist()
print(missions[:5])
print("average price:", round(sum(prices) / len(prices), 2), "($M)")

## Write Parquet

`write_parquet` turns a morsel into Parquet bytes (zstd-compressed by default). Read it back to check it round-trips.

In [ ]:
data = parquet.write_parquet(spacex)
with open("spacex.parquet", "wb") as f:
    f.write(data)

print(len(data), "bytes;", parquet.read_metadata("spacex.parquet").num_rows, "rows read back")

## Stream batches at constant memory

`open_parquet_writer` writes one row group per call to `write_row_group`, passing bytes to your sink as it goes, so a large conversion never holds the whole file in memory.

In [ ]:
with open("successes.parquet", "wb") as f:
    with parquet.open_parquet_writer(f.write) as writer:
        with parquet.read_parquet(
            "space_missions.parquet", predicates=[("Mission_Status", "==", "Success")]
        ) as reader:
            for morsel in reader:
                writer.write_row_group(morsel)

print(parquet.read_metadata("successes.parquet").num_rows, "successful launches written")